# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² mass spectrometry protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset is referenced using its Croissant schema URL and includes multiple record sets with fields and columns suitable for scientific exploration and machine learning tasks.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets and their fields by their `@id`s.

In [ ]:
# List all record sets and fields defined in the Croissant metadata

from typing import Any

def display_recordsets(ds: mlc.Dataset) -> None:
    rs_list = ds.metadata.dict().get('recordSet', [])
    if not rs_list:
        print("No top-level record sets found. Trying to extract from other metadata...")
        # Search for attached record sets
        # FIXME: As last fallback, fetch record set IDs from croissant content dictionary
        for k, v in ds.metadata.dict().items():
            if isinstance(v, list):
                for item in v:
                    if isinstance(item, dict) and item.get('@type') == 'RecordSet':
                        print(f"Found RecordSet @id: {item['@id']}, name: {item.get('name','')}")
        return
    for rs in rs_list:
        print(f"RecordSet @id: {rs.get('@id')}, name: {rs.get('name','')}")

def display_fields(ds: mlc.Dataset, record_set_id: str) -> None:
    # Fetch a preview and collect field keys
    try:
        records = list(ds.records(record_set=record_set_id))
        if records:
            print(f"Fields in record set {record_set_id} (by keys of sample record):")
            print(list(records[0].keys()))
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error fetching records for {record_set_id}: {e}")

print("Discovering record sets...")
# This dataset does not declare record sets at top level. Instead, we probe for known ones.
# FAIR^2 datasets use annotated IDs; known record set for the main table is: 'https://api.app.sen.science/frontiers/7154140/5ffcd67f-6a96-4e50-b7ce-8f93f8b010ad' (example, needs fetching)
all_possible_rs = []
for k, v in dataset.metadata.dict().items():
    if isinstance(v, list):
        for item in v:
            if isinstance(item, dict) and item.get('@type') == 'RecordSet':
                all_possible_rs.append(item['@id'])
if not all_possible_rs:
    # Guess based on standard FAIR² structure
    all_possible_rs = ['https://api.app.sen.science/frontiers/7154140/5ffcd67f-6a96-4e50-b7ce-8f93f8b010ad']

print(f"Discovered record set IDs: {all_possible_rs}")
for rsid in all_possible_rs:
    print(f"Exploring fields in RecordSet {rsid}:")
    try:
        records = list(dataset.records(record_set=rsid))
        fields = list(records[0].keys()) if records else []
        print(f"Fields (@id): {fields}")
    except Exception as exc:
        print(f"Could not access records for {rsid}: {exc}")

## 3. Data Extraction
Load data from the discovered record set(s) into Pandas DataFrames for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Main RecordSet from metadata or previous cell (use @id)
record_sets = all_possible_rs
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame loaded for RecordSet {record_set_id} with columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For further analysis, we pick the first discovered record set:
main_record_set_id = record_sets[0]
df_main = dataframes[main_record_set_id]
print(f"Exploring columns in main DataFrame: {df_main.columns.tolist()}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Some typical analysis includes filtering, normalization, and grouping by important attributes (fields/columns by `@id`).

In [ ]:
# -- Example EDA --
# Identify available numeric fields
numeric_fields = df_main.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields available (@id): {numeric_fields}")

# If available, let's choose 'cr:peptideCount' (replace if not present)
if 'cr:peptideCount' in df_main.columns:
    numeric_field_id = 'cr:peptideCount'
elif numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError("No numeric fields found in this record set!")

print(f"Using numeric field for EDA: {numeric_field_id}")
# Set a threshold for filtering
threshold = 10
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (first 5):")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records (first 5):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Identify a group field (e.g., 'cr:accession' or similar; replace as appropriate)
group_field_id = None
possible_group_fields = ['cr:accession', 'cr:modification', 'cr:sampleId']
for field in possible_group_fields:
    if field in df_main.columns:
        group_field_id = field
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}, average {numeric_field_id} (first 5):")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions, such as the normalized numeric field or relationships between key attributes.

In [ ]:
import matplotlib.pyplot as plt

# Distribution of the numeric field
plt.figure(figsize=(8,5))
filtered_df[numeric_field_id].hist(bins=30)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping was possible, plot means by group
if group_field_id:
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(12,5))
    plt.bar(top_groups[group_field_id], top_groups[numeric_field_id])
    plt.title(f"Top 10 groups by mean {numeric_field_id} (grouped by {group_field_id})")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a FAIR² mass spectrometry protein dataset using `mlcroissant`. We reviewed available record sets and fields by their `@id`, loaded the main data into Pandas for analysis, applied basic filtering and normalization to a numeric field, and visualized the results.

To go further, try:
- Exploring additional record sets/fields by their `@id`
- Applying domain-specific analysis (e.g., protein quantification, outlier detection, sequence-based grouping)
- Integrating this dataset with others for comparative proteomics

Explore the full Croissant schema for more advanced operations!